# step 1-Olist E-commerce Business Analysis

## Project Objective

The objective of this project is to analyze Olist's e-commerce data to understand sales performance, customer behavior, product performance, seller performance, delivery efficiency, payment trends, and customer satisfaction. The insights generated will help the business make data-driven decisions to improve revenue, customer experience, and operational efficiency.

## step 2-Import Required Libraries

In this section, we import the Python libraries required for data manipulation, numerical operations, and data visualization.

In [196]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns
pd.set_option("display.max_columns", None)

## # 3. Load Datasets

## Objective

The Olist dataset is divided into multiple tables because it follows a relational database structure. Each table stores a different type of business information, such as customers, orders, products, sellers, payments, and reviews.

Before performing any analysis, all datasets need to be loaded into Python as Pandas DataFrames.

In [197]:
customers = pd.read_csv("../data/raw_data/olist_customers_dataset.csv")

orders = pd.read_csv("../data/raw_data/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw_data/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw_data/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw_data/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw_data/olist_products_dataset.csv")

sellers = pd.read_csv("../data/raw_data/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/raw_data/olist_geolocation_dataset.csv")

category_translation = pd.read_csv("../data/raw_data/product_category_name_translation.csv")

# 4. Data Understanding

## Objective

Before cleaning or merging the datasets, it is important to understand their structure and quality.

In this phase, we will examine all datasets to identify:

- Number of rows and columns
- Data types
- Missing values
- Duplicate records

This provides a high-level overview of the data and helps determine the cleaning steps required before analysis.

### 4.1 Store All Datasets

To avoid writing repetitive code, all DataFrames are stored in a dictionary.

This allows the same analysis to be applied to every dataset using a simple loop, making the notebook cleaner and easier to maintain.

In [198]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

### 4.2 Dataset Summary

Before cleaning the data, it is important to get an overview of all datasets.

The following summary provides:

- Number of rows
- Number of columns

This helps understand the size of each dataset and identifies which tables contain the most information.

## check the structure of dataset 

In [199]:
summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

summary_df = pd.DataFrame(summary)

summary_df

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order Items,112650,7
3,Payments,103886,5
4,Reviews,99224,7
5,Products,32951,9
6,Sellers,3095,4
7,Geolocation,1000163,5
8,Category Translation,71,2


## check data quality of dataset 

## 4.3 Missing Value Analysis

Missing values can affect calculations, joins, and business analysis.

We will identify missing values in each column across all Olist datasets before deciding how to handle them.

In [200]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"Missing Values - {name}")
    print("=" * 60)
    print(df.isnull().sum())


Missing Values - Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing Values - Orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing Values - Order Items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Missing Values - Payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Missing Values - Reviews
review_id                      0
order_id            

## 4.4 Duplicate Analysis

Duplicate records can cause the same information to be counted more than once.

We will check whether completely identical rows exist in each dataset before performing data cleaning.

In [201]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"Duplicate Rows - {name}")
    print("=" * 60)
    print("Duplicate rows:", df.duplicated().sum())


Duplicate Rows - Customers
Duplicate rows: 0

Duplicate Rows - Orders
Duplicate rows: 0

Duplicate Rows - Order Items
Duplicate rows: 0

Duplicate Rows - Payments
Duplicate rows: 0

Duplicate Rows - Reviews
Duplicate rows: 0

Duplicate Rows - Products
Duplicate rows: 0

Duplicate Rows - Sellers
Duplicate rows: 0

Duplicate Rows - Geolocation
Duplicate rows: 261831

Duplicate Rows - Category Translation
Duplicate rows: 0


In [202]:
geolocation.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP


In [203]:
geolocation.shape

(1000163, 5)

## realtionship 

## 4.5 Primary Key and Foreign Key Validation

The Olist dataset contains multiple related tables. Before merging them, we need to understand the key columns that connect the tables.

We will check the uniqueness of important identifier columns to understand the relationships between tables and avoid incorrect joins or duplicated records.

## customeres table 

In [204]:
customers["customer_id"].nunique()

99441

In [205]:
customers.shape[0]

99441

## orders table

In [206]:
orders["order_id"].nunique()

99441

In [207]:
orders.shape[0]

99441

## order_items table

In [208]:
order_items["order_id"].nunique()

98666

In [209]:
order_items.shape[0]

112650

### Order Items Key Validation

The `order_id` column contains 98,666 unique values across 112,650 rows.

Unlike the Orders table, `order_id` is not unique in the Order Items table because a single order can contain multiple items.

Therefore, `order_id` represents a one-to-many relationship between Orders and Order Items.

In [210]:
order_items["order_id"].nunique()

98666

In [211]:
orders["order_id"].nunique()

99441

In [212]:
missing_order_ids = set(orders["order_id"]) - set(order_items["order_id"])

len(missing_order_ids)

775

### Order ID Relationship: Orders and Order Items

The Orders table contains 99,441 unique `order_id` values, while the Order Items table contains 98,666 unique `order_id` values.

This indicates that not every order in the Orders table has a corresponding record in the Order Items table.

The difference of 775 order IDs will be investigated further before deciding how these records should be handled.

The `order_id` is unique in the Orders table, while it can appear multiple times in Order Items because one order can contain multiple items.

In [213]:
order_items["product_id"].nunique()

32951

In [214]:
products["product_id"].nunique()

32951

In [215]:
set(order_items["product_id"].unique()) == set(products["product_id"].unique())

True

### Product Key Validation

The unique `product_id` values in the Order Items and Products tables were compared.

The sets of product IDs are identical, confirming that the two tables share the same product identifiers and can be connected using `product_id`.

A product can appear multiple times in Order Items because the same product may be purchased in multiple orders.

In [216]:
order_items["seller_id"].nunique()

3095

In [217]:
sellers["seller_id"].nunique()

3095

In [218]:
set(order_items["seller_id"].unique()) == set(sellers["seller_id"].unique())

True

## seller key valiadation 
the unique seller_id in both sellers and order_items after compared

## specific issues

## 4.6 Date Column Validation

Date and time fields are important for analyzing sales trends, order processing, and delivery performance.

We will inspect the date columns in the Orders table and verify their current data types before cleaning.

In [219]:
orders[[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].dtypes

order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

### Date Column Observation

The date-related columns in the Orders table are currently stored as `object` data types.

Since these columns represent dates and timestamps, they should be converted to an appropriate datetime format during the Data Cleaning phase.

This conversion will allow date-based analysis such as monthly sales trends, delivery-time calculations, and order processing analysis.

In [220]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Order Status Observation

The Orders table contains multiple order statuses, with `delivered` being the dominant status.

Other statuses include `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created`, and `approved`.

These statuses will be retained during data cleaning because each represents a different stage of the order lifecycle. They will be considered appropriately during business analysis.

# 5. Data Cleaning

The objective of this phase is to identify and handle data-quality issues found during data understanding.

The cleaning process will include:
- Converting incorrect data types
- Handling missing values
- Handling duplicate records
- Standardizing text values where required
- Validating the cleaned data

## let's check missing values 
before converting them

In [221]:
orders[[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].isnull().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## convert the columns into date_time data types

In [222]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [223]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [224]:
orders[date_columns].isnull().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Date Conversion

The order date columns were converted from `object` to `datetime64[ns]` using `pd.to_datetime()`.

Missing date values were retained because they can represent legitimate stages of the order lifecycle, such as orders that were not approved, shipped, or delivered.

The converted datetime fields can now be used for time-based analysis and delivery-time calculations.

## 5.2 Missing Value Analysis

Missing values were reviewed across all Olist tables.

Missing values are not automatically removed or replaced because their meaning depends on the business context. For example, missing delivery dates may be legitimate for orders that were not delivered.

Each important missing-value field will therefore be evaluated before applying a cleaning treatment.

In [225]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"Missing Values - {name}")
    print("=" * 60)

    missing = df.isnull().sum()
    print(missing[missing > 0])


Missing Values - Customers
Series([], dtype: int64)

Missing Values - Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Missing Values - Order Items
Series([], dtype: int64)

Missing Values - Payments
Series([], dtype: int64)

Missing Values - Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

Missing Values - Products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Missing Values - Sellers
Series([], dtype: int64)

Missing Values - Geolocation
Series([], dtype: int64)

Missing Values - Category Translation
Series([], dtype: int64)


### Orders - Missing Date Values

The missing values in the order date columns were retained because they can represent legitimate stages of the order lifecycle.

For example, an order that was not delivered may not have a `order_delivered_customer_date`.

Replacing these values with artificial dates could distort delivery-time and operational analysis.

### Reviews - Missing Comment Values

The review comment fields contain a large number of missing values.

These were retained because customers can submit a review score without providing written comments. Therefore, the missing values do not necessarily indicate invalid records.

The review score remains available for quantitative analysis, while written comments can be analyzed separately when available.

In [226]:
products[products["product_category_name"].isnull()].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [227]:
products[products["product_category_name"].isnull()].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [228]:
products["product_category_name"] = products["product_category_name"].fillna("Unknown")

In [229]:
products["product_category_name"].isnull().sum()

np.int64(0)

In [230]:
products["product_category_name"].value_counts().head()

product_category_name
cama_mesa_banho          3029
esporte_lazer            2867
moveis_decoracao         2657
beleza_saude             2444
utilidades_domesticas    2335
Name: count, dtype: int64

### Product Category Missing Values - Validation

The 610 missing `product_category_name` values were replaced with `Unknown`.

The records were retained because they contain valid `product_id` values and other product information. No category was inferred or artificially assigned.

In [231]:
products[
    products["product_weight_g"].isnull() |
    products["product_length_cm"].isnull() |
    products["product_height_cm"].isnull() |
    products["product_width_cm"].isnull()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Product Physical Attribute Missing Values

Two product records contain missing physical attributes such as weight and dimensions.

The records were retained because they contain valid product IDs and other product information. The missing numerical measurements were not replaced with zero, mean, or arbitrary values because doing so could introduce inaccurate physical information.

The missing values will remain as NaN.

## text/data standardization

In [232]:
products["product_category_name"].nunique()

74

In [233]:
products["product_category_name"].value_counts().head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
Unknown                               610
eletronicos                           517
construcao_ferramentas_construcao     400
Name: count, dtype: int64

## Check Numerical Values
check important numerical columns contain negative values or impossible values

In [234]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


### Numerical Value Validation - Order Items

The `price` and `freight_value` columns were checked for invalid negative values.

No negative values were found. A freight value of zero was considered valid because it can represent free shipping.

Although some high-value prices exist, they were retained because an unusually high value is not necessarily an error. These values will be considered during exploratory and business analysis.

In [235]:
payments[["payment_value", "payment_installments"]].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


### Numerical Value Validation - Payments

The `payment_value` and `payment_installments` columns were checked for invalid negative values.

No negative values were identified. Zero values were retained because they were not automatically assumed to represent invalid records.

The maximum payment value of 13,664.08 was also retained because an unusually high value is not necessarily an error and requires business context before being classified as invalid.

## Final Data Cleaning Validation

In [236]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)


### Final Table Shape Validation

The cleaned Olist tables were checked to confirm their row and column counts after the data-cleaning process.

The table sizes remained consistent with the expected structure of the dataset. The higher number of records in `order_items` and `payments` compared with `orders` is expected because a single order can contain multiple order items and multiple payment records.

No unexpected large-scale row loss was observed during cleaning.

## Missing values after cleaning

In [237]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    
    print("\n" + "=" * 60)
    print(f"Remaining Missing Values - {name}")
    print("=" * 60)
    print(missing[missing > 0])


Remaining Missing Values - Customers
Series([], dtype: int64)

Remaining Missing Values - Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Remaining Missing Values - Order Items
Series([], dtype: int64)

Remaining Missing Values - Payments
Series([], dtype: int64)

Remaining Missing Values - Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

Remaining Missing Values - Products
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Remaining Missing Values - Sellers
Series([], dtype: int64)

Remaining Missing Values - Geolocation
Series([], dtype: int64)

Remaining Missing Values - Category Translation
Series([], dtype: int64)


## 5.6 Final Data Cleaning Validation

After applying the required cleaning steps, the datasets were revalidated for row counts, missing values, data types, and key data-quality issues.

The remaining missing values were reviewed and intentionally retained where they represented unavailable or legitimate information rather than errors.

No major invalid numerical values or unexpected row loss were identified.

The datasets are now ready for the data transformation and analytical phase.

## step-6 Data Transformation(ORDER METRICS)

## 6.1 Create delivery_days

In [238]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [239]:
orders[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


### 6.1 Delivery Time

A new `delivery_days` column was created to measure the number of days between order purchase and customer delivery.

This metric will be used later to analyze delivery performance, identify delayed orders, and compare delivery performance across locations and other business dimensions.

Orders without a recorded delivery date retain a missing `delivery_days` value because their actual delivery duration cannot be determined.

## 6.2 create delivey_delay_days

In [240]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [241]:
orders[[
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days"
]].head(10)

,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,2017-10-10 21:25:13,2017-10-18,-8.0
1,2018-08-07 15:27:45,2018-08-13,-6.0
2,2018-08-17 18:06:29,2018-09-04,-18.0
3,2017-12-02 00:28:42,2017-12-15,-13.0
4,2018-02-16 18:17:02,2018-02-26,-10.0
5,2017-07-26 10:57:55,2017-08-01,-6.0
6,NaT,2017-05-09,NaN
7,2017-05-26 12:55:51,2017-06-07,-12.0
8,2017-02-02 14:08:10,2017-03-06,-32.0
9,2017-08-16 17:14:30,2017-08-23,-7.0


In [242]:
orders["delivery_delay_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

## 6.3 Delivery_status 

In [243]:
import numpy as np

orders["delivery_status"] = np.select(
    [
        orders["delivery_delay_days"] < 0,
        orders["delivery_delay_days"] == 0,
        orders["delivery_delay_days"] > 0
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default="Not Delivered"
)

In [244]:
orders["delivery_status"].value_counts()

delivery_status
Early            88649
Late              6535
Not Delivered     2965
On Time           1292
Name: count, dtype: int64

### 6.3 Delivery Status

Orders were classified into four delivery-status groups based on the difference between actual and estimated delivery dates:

- Early: delivered before the estimated date
- On Time: delivered on the estimated date
- Late: delivered after the estimated date
- Not Delivered: no actual customer delivery date was available

The resulting categories cover all 99,441 orders and will be used later for delivery-performance analysis and dashboard visualizations.

## 6.4 order date
# order_year

In [245]:
orders["order_year"] = orders["order_purchase_timestamp"].dt.year

In [246]:
orders["order_year"].value_counts().sort_index()

order_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

# order_month

In [247]:
orders["order_month"] = orders["order_purchase_timestamp"].dt.month

In [248]:
orders["order_month"].value_counts().sort_index()

order_month
1      8069
2      8508
3      9893
4      9343
5     10573
6      9412
7     10318
8     10843
9      4305
10     4959
11     7544
12     5674
Name: count, dtype: int64

# order_month_name

In [249]:
orders["order_month_name"] = orders["order_purchase_timestamp"].dt.month_name()

In [250]:
orders[["order_month", "order_month_name"]].drop_duplicates().sort_values("order_month")

,order_month,order_month_name
8,1,January
4,2,February
14,3,March
6,4,April
7,5,May
12,6,June
1,7,July
2,8,August
20,9,September
0,10,October


## Create Revenue Metric
# Total item value = Price + Freight

In [251]:
order_items["item_total"] = (
    order_items["price"] + order_items["freight_value"]
)

In [252]:
order_items[
    ["price", "freight_value", "item_total"]
].head()

,price,freight_value,item_total
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


### 6.4 Order Item Value

A new `item_total` column was created by combining the product price and freight value.

This metric represents the total value associated with an individual order item, including the product price and shipping charge.

It will be used later for sales and order-value analysis.

## 6.5 Create Order-Level Value
# We created item_total in Order Items.

In [253]:
order_value = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("item_total", "sum"),
        item_count=("order_item_id", "count")
    )
)

In [254]:
order_value.head()

,order_id,order_value,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1


In [255]:
order_value.shape

(98666, 3)

In [256]:
order_value.shape

(98666, 3)

## Add Order_value to Orders

In [257]:
orders = orders.merge(
    order_value,
    on="order_id",
    how="left"
)

In [258]:
orders.shape

(99441, 16)

In [259]:
orders[[
    "order_id",
    "order_value",
    "item_count"
]].head()

,order_id,order_value,item_count
0,e481f51cbdc54678b7cc49136f2d6af7,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,28.62,1.0


### 6.5 Order-Level Value

Since one order can contain multiple order items, item-level values were aggregated to the order level.

Two metrics were created:

- `order_value`: total product and freight value associated with the order
- `item_count`: number of items in the order

These metrics were then merged into the Orders table using `order_id`, while preserving all orders through a left join.

## customer metrics

In [260]:
customer_metrics = (
    orders
    .groupby("customer_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("order_value", "sum")
    )
)

In [261]:
customer_metrics.shape

(99441, 3)

In [262]:
customer_metrics.head()

,customer_id,total_orders,total_spent
0,00012a2ce6f8dcda20d059ce98491703,1,114.74
1,000161a058600d5901f007fab4c27140,1,67.41
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42
3,0002414f95344307404f0ace7a26f1d5,1,179.35
4,000379cdec625522490c315e70c7a9fb,1,107.01


In [263]:
customers["customer_id"].nunique()

99441

In [264]:
customers["customer_unique_id"].nunique()

96096

## Use customer_unique_id

We just learned that customer_id is not the best identifier for actual customer-level analysis.

## Create Correct Customer Metrics
add unique_customer_id to order_customer table 

In [265]:
orders_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

In [266]:
orders_customer.shape

(99441, 17)

## Customer-Level Analysis

In [267]:
customer_metrics = (
    orders_customer
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("order_value", "sum")
    )
)

In [268]:
customer_metrics.shape

(96096, 3)

In [269]:
customer_metrics.head()

,customer_unique_id,total_orders,total_spent
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89


### 6.9 Customer-Level Metrics

Customer-level metrics were created using `customer_unique_id`, which represents the actual unique customer.

The metrics include:

- `total_orders`: number of unique orders placed by the customer
- `total_spent`: total value associated with the customer's orders

Using `customer_unique_id` prevents customers with multiple customer records from being treated as separate customers.

## Step 6.10 — Average Order Value (AOV)

In [270]:
customer_metrics["average_order_value"] = (
    customer_metrics["total_spent"] /
    customer_metrics["total_orders"]
)

In [271]:
customer_metrics.head()

,customer_unique_id,total_orders,total_spent,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89


## Step 6.11 — Customer Purchase Frequency

In [272]:
import numpy as np

customer_metrics["customer_type"] = np.select(
    [
        customer_metrics["total_orders"] == 1,
        customer_metrics["total_orders"] > 1
    ],
    [
        "One-time Customer",
        "Repeat Customer"
    ],
    default="Unknown"
)

In [273]:
customer_metrics["customer_type"].value_counts()

customer_type
One-time Customer    93099
Repeat Customer       2997
Name: count, dtype: int64

## Step 6.12 — Customer Spending Segmentation

In [274]:
customer_metrics["spending_segment"] = pd.qcut(
    customer_metrics["total_spent"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

In [275]:
customer_metrics["spending_segment"].value_counts()

spending_segment
Low          24033
Very High    24024
High         24021
Medium       24018
Name: count, dtype: int64

## Step 6.13 — Create a Product Revenue Metric

In [276]:
order_items_products = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

In [277]:
order_items_products[
    ["order_id", "product_id", "item_total", "product_category_name"]
].head()

,order_id,product_id,item_total,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,72.19,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,259.83,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,216.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,25.78,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,218.04,ferramentas_jardim


## Step 6.14 — Category-Level Sales

In [278]:
category_sales = (
    order_items_products
    .groupby("product_category_name", as_index=False)
    .agg(
        total_sales=("item_total", "sum"),
        total_items=("order_item_id", "count")
    )
    .sort_values("total_sales", ascending=False)
)

In [279]:
category_sales.head(10)

,product_category_name,total_sales,total_items
12,beleza_saude,1441248.07,9670
67,relogios_presentes,1305541.61,5991
14,cama_mesa_banho,1241681.72,11115
33,esporte_lazer,1156656.48,8641
45,informatica_acessorios,1059272.40,7827
55,moveis_decoracao,902511.79,8334
73,utilidades_domesticas,778397.77,6964
27,cool_stuff,719329.95,3796
9,automotivo,685384.32,4235
41,ferramentas_jardim,584219.21,4347


## Step 6.15 — Seller Performance

In [280]:
seller_sales = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        total_sales=("item_total", "sum"),
        total_items=("order_item_id", "count"),
        total_orders=("order_id", "nunique")
    )
    .sort_values("total_sales", ascending=False)
)

## Step 6.16 — Payment Analysis

In [281]:
payment_summary = (
    payments
    .groupby("payment_type", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        number_of_payments=("payment_value", "count"),
        average_payment_value=("payment_value", "mean")
    )
    .sort_values("total_payment_value", ascending=False)
)

## Step 6.17 — Monthly Order Analysis

In [282]:
monthly_orders = (
    orders
    .groupby(
        ["order_year", "order_month", "order_month_name"],
        as_index=False
    )
    .agg(
        total_orders=("order_id", "nunique"),
        total_order_value=("order_value", "sum")
    )
    .sort_values(["order_year", "order_month"])
)

In [283]:
monthly_orders.head(15)

,order_year,order_month,order_month_name,total_orders,total_order_value
0,2016,9,September,4,354.75
1,2016,10,October,324,56808.84
2,2016,12,December,1,19.62
3,2017,1,January,800,137188.49
4,2017,2,February,1780,286280.62
5,2017,3,March,2682,432048.59
6,2017,4,April,2404,412422.24
7,2017,5,May,3700,586190.95
8,2017,6,June,3245,502963.04
9,2017,7,July,4026,584971.62


## data validation 

In [284]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [285]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [286]:
products["product_id"].duplicated().sum()

np.int64(0)

In [287]:
(order_items["price"] < 0).sum()

np.int64(0)

In [288]:
(order_items["freight_value"] < 0).sum()

np.int64(0)

In [289]:
(orders["order_value"] < 0).sum()

np.int64(0)

In [290]:
(orders["delivery_days"] < 0).sum()

np.int64(0)

## export clean_data 

In [295]:
orders.to_csv("data/processed/orders.csv", index=False)
customers.to_csv("data/processed/customers.csv", index=False)
order_items.to_csv("data/processed/order_items.csv", index=False)
payments.to_csv("data/processed/payments.csv", index=False)
reviews.to_csv("data/processed/reviews.csv", index=False)
products.to_csv("data/processed/products.csv", index=False)
sellers.to_csv("data/processed/sellers.csv", index=False)
category_translation.to_csv("data/processed/category_translation.csv", index=False)

## get copy of base table of orders 

In [ ]:
orders_sql = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()

In [ ]:
orders_sql.shape

(99441, 8)

In [ ]:
orders_sql.to_csv(
    "data/processed/orders_sql.csv",
    index=False
)